In [ ]:

from getpass import getpass
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_histogram

# PART 1: Connect to IBM Quantum
api_key = getpass("Enter your IBM Quantum API key: ")
instance = input("Enter your IBM Quantum instance / CRN: ").strip()

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=api_key,
    instance=instance
)

print("Connected successfully!")

# PART 2: Select a real quantum computer
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=2
)

print("Selected backend:", backend.name)

# PART A: CORRELATED OUTCOMES
# Expected outcomes: 00 and 11
qc_a = QuantumCircuit(2)
qc_a.h(0)
qc_a.cx(0, 1)
qc_a.measure_all()

# PART B: OPPOSITE OUTCOMES
# Add one X gate to flip the second qubit
# Expected outcomes: 01 and 10
qc_b = QuantumCircuit(2)
qc_b.h(0)
qc_b.cx(0, 1)
qc_b.x(1)
qc_b.measure_all()

# Display circuits
print("\nPART A: Correlated Circuit")
display(qc_a.draw("mpl"))

print("\nPART B: Opposite Outcomes Circuit")
display(qc_b.draw("mpl"))

# Predictions
print("\nPREDICTIONS")
print("Part A: 00 and 11 should dominate.")
print("Part B: 01 and 10 should dominate.")
print("Ideal probability of each dominant outcome: 50%.")

# Transpile for IBM Quantum hardware
pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1
)

isa_circuits = pm.run([qc_a, qc_b])

# Execute both circuits for 100 shots each
sampler = Sampler(mode=backend)
job = sampler.run(isa_circuits, shots=100)

print("\nJob ID:", job.job_id())
result = job.result()

# Get results
counts_a = result[0].data.meas.get_counts()
counts_b = result[1].data.meas.get_counts()

print("\nPART A RESULTS (100 shots):")
print(counts_a)

print("\nPART B RESULTS (100 shots):")
print(counts_b)

# Histograms
plot_histogram(
    [counts_a, counts_b],
    legend=["Part A: Correlated", "Part B: Opposite"],
    title="Challenge 3: Correlated vs Opposite Outcomes"
)
plt.show()

# Compare actual results
def compare_results(name, counts):
    total = sum(counts.values())
    print(f"\n{name}")
    for outcome in ["00", "01", "10", "11"]:
        number = counts.get(outcome, 0)
        print(f"{outcome}: {number} shots ({number / total * 100:.1f}%)")

compare_results("PART A", counts_a)
compare_results("PART B", counts_b)


Enter your IBM Quantum API key:  ········
Enter your IBM Quantum instance / CRN:  crn:v1:bluemix:public:quantum-computing:us-east:a/00b3915f8e7243f7a01812846e4b3fa3:c5c6654f-1782-4b1a-9ec3-6b7cea9475b3::
